# CodeSprint AI: Multi-Model Python Performance Optimizer

Paste slow Python → fan out to multiple LLMs → translate to C++ or Rust → compile, run, validate output, report speedup.

Extends the Week 4 lab with:
1. Rust as a second target language
2. Multi-model fan-out in one click
3. Output validation (exact + numeric tolerance)
4. A leaderboard table in the UI


In [3]:
# Imports

import os, io, sys, time, subprocess, re
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr


/Users/mdqamarhashmi/Library/Python/3.12/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load environment variables from .env file
load_dotenv(override=True)

openai     = OpenAI()
anthropic  = OpenAI(api_key=os.getenv('ANTHROPIC_API_KEY'),  base_url="https://api.anthropic.com/v1/")
gemini     = OpenAI(api_key=os.getenv('GOOGLE_API_KEY'),     base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
grok       = OpenAI(api_key=os.getenv('GROK_API_KEY'),       base_url="https://api.x.ai/v1")
groq       = OpenAI(api_key=os.getenv('GROQ_API_KEY'),       base_url="https://api.groq.com/openai/v1")
openrouter = OpenAI(api_key=os.getenv('OPENROUTER_API_KEY'), base_url="https://openrouter.ai/api/v1")

# Define the models and their corresponding clients
models = [
    "gpt-5",
    "claude-sonnet-4-5-20250929",
    "gemini-2.5-pro",
    "grok-4",
    "openai/gpt-oss-120b",
    "qwen/qwen3-coder-30b-a3b-instruct",
]

clients = {
    "gpt-5": openai,
    "claude-sonnet-4-5-20250929": anthropic,
    "gemini-2.5-pro": gemini,
    "grok-4": grok,
    "openai/gpt-oss-120b": groq,
    "qwen/qwen3-coder-30b-a3b-instruct": openrouter,
}

In [5]:
# C++ : compile commands and run commands
cpp_compile  = ["clang++", "-std=c++17", "-O3", "-DNDEBUG", "main.cpp", "-o", "main_cpp"]
cpp_run      = ["./main_cpp"]

# Rust — compile commands and run commands
rust_compile = ["rustc", "--edition", "2021", "-O", "main.rs", "-o", "main_rs"]
rust_run     = ["./main_rs"]


In [ ]:
# System and User Prompts

def system_prompt(target):
    lang = "C++17" if target == "cpp" else "Rust (edition 2021, no external crates)"
    return f"""Convert Python to {lang} that produces byte-identical stdout and runs as fast as possible.
Respond with code only - no markdown fences, no explanation."""

def user_prompt(python, target):
    return f""" Port the following Python code to {'C++' if target == "cpp" else "Rust"}. Match stdout exactly (formatting, decimals, newlines). Single self-contained file. Stdlib only.

```python
{python}
```"""


# Generate messages for the LLMs
def messages_for_llms(python, target):
    return [
        {"role": "system", "content": system_prompt(target)},
        {"role": "user",   "content": user_prompt(python, target)},
    ]

In [ ]:
# Write the generated code to a file
def write_source(code, target):
    filename = "main.cpp" if target == "cpp" else "main.rs"
    with open(filename, "w") as f:
        f.write(code)

# Translate the Python code to the target language using the specified model
def translate(model, python, target):
    client = clients[model]
    reasoning_effort = "high" if "gpt" in model else None
    t0 = time.perf_counter()
    resp = client.chat.completions.create(
        model=model,
        messages=messages_for_llms(python, target),
        reasoning_effort=reasoning_effort,
    )
    elapsed = time.perf_counter() - t0
    code = resp.choices[0].message.content
    for fence in ["```", "```cpp", "```rust"]:
        code = code.replace(fence, "")
    code = code.strip()
    write_source(code, target)
    return code, elapsed